In [ ]:
# ===================================================================================
# GP SERVICE INVENTORY SCRIPT (READ-ONLY CATALOG)
# - GP Services cannot be migrated via API; they must be republished from
#   source toolboxes on the new ArcGIS Server.
# - This script catalogs all GP Services with rebuild recommendations.
# ===================================================================================

import pandas as pd
import json
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
GP_INVENTORY_CSV = os.path.join(os.path.dirname(LOG_FILE), "gp_service_inventory.csv")

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_16_gp_services.json")
if os.path.exists(_sidecar_path):
    import json as _json
    GP_SERVICE_IDS = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(GP_SERVICE_IDS)} GP Service IDs from sidecar.")
else:
    GP_SERVICE_IDS = [
        # Example Source IDs
        # "abc123def456...",
    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)

    print(f"   > Source Connected: {source_gis.url}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    sys.exit(1)

# =============================================================================
# --- LOGGING ------------------------------------------------------------------
# =============================================================================
STATS = {"scanned": 0, "inventoried": 0}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} items previously migrated.")
    except: pass
else:
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def log_migration(source_id, index, target_id, title, item_type):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, index, target_id, title,
                datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item_type
            ])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# --- INITIALIZE INVENTORY CSV ---
if not os.path.exists(GP_INVENTORY_CSV):
    with open(GP_INVENTORY_CSV, "w", newline="") as f:
        csv.writer(f).writerow([
            "Title", "SourceID", "Owner", "URL", "Tasks",
            "Description", "Recommendation"
        ])
    print(f"Created inventory CSV: {GP_INVENTORY_CSV}")

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"\nStarting GP Service Inventory...")
start_time = time.time()

for item_id in GP_SERVICE_IDS:
    try:
        STATS["scanned"] += 1

        source_item = source_gis.content.get(item_id)
        if not source_item:
            print(f"❌ Source {item_id} not found.")
            continue

        print(f"\nInventorying: {source_item.title} ({item_id})...")

        # --- GATHER PROPERTIES ---
        title = source_item.title
        owner = source_item.owner
        svc_url = source_item.url or ""
        item_type = source_item.type
        type_keywords = list(source_item.typeKeywords) if source_item.typeKeywords else []
        description = source_item.description or ""

        # --- TRY TO GET SERVICE TASKS ---
        tasks = []
        try:
            if svc_url:
                res = requests.get(f"{svc_url}?f=json&token={SOURCE_TOKEN}", verify=False)
                svc_info = res.json()
                tasks = [t.get("name", "Unknown") for t in svc_info.get("tasks", [])]
        except: pass

        tasks_str = ", ".join(tasks) if tasks else "(none detected)"
        print(f"   📋 Owner: {owner}")
        print(f"   🔗 URL: {svc_url}")
        print(f"   🔧 Tasks: {tasks_str}")

        # --- WRITE TO INVENTORY CSV ---
        recommendation = "Republish from source toolbox on target ArcGIS Server"
        with open(GP_INVENTORY_CSV, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([
                title, item_id, owner, svc_url, tasks_str,
                description[:500], recommendation
            ])

        # --- LOG TO MIGRATION HISTORY ---
        log_migration(item_id, "N/A", "INVENTORY_ONLY", title, "Geoprocessing Service")
        STATS["inventoried"] += 1
        print(f"   ✅ Inventoried: {title}")

    except Exception as e:
        print(f"❌ FAILED on {item_id}: {e}")

# =============================================================================
# --- REPORT -------------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("        GP SERVICE INVENTORY REPORT        ")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 📦 Scanned:                   {STATS['scanned']}")
print(f" 📋 Inventoried:               {STATS['inventoried']}")
print("-" * 60)
print(f" 📄 Inventory CSV:             {GP_INVENTORY_CSV}")
print("-" * 60)
print(f" ⚠️  NOTE: GP Services must be republished manually from source toolboxes.")
print(f"    This script catalogs them for planning purposes only.")
print("=" * 60)